# Notebook 04b: LeadwiseTransformer — Cross-Lead ECG Classification

**Novel contribution of this project.** Every other model in this pipeline treats the 12 ECG leads as independent channels (channel-independent). `LeadwiseTransformer` instead models explicit *inter-lead relationships* — the architectural insight that mirrors clinical practice:

> A cardiologist diagnosing inferior MI does not read lead II in isolation — they observe concordant ST elevation across II, III, and aVF simultaneously.

## Architecture

```
Input (B, 12, 1000)
  |
  +-- Conv1d patch embedding (per lead, shared weights)
  |
  +-- Temporal Transformer encoder  (3 layers, Pre-LN, shared across leads)
  |     Captures within-lead temporal patterns
  |
  +-- Cross-lead Multi-head Attention  <-- novel part
  |     Q = K = V = lead token sequence
  |     Each lead attends to all other leads
  |
  +-- Mean pool over leads
  |
  +-- Classifier head  -> (B, 5) logits
```

Total parameters: ~476 K (trained from scratch — no pretrained weights).

This notebook trains the **LoRA r=8** variant. Results are compared against HuBERT baselines in `04c_comparison.ipynb`.

In [ ]:
import sys, os, warnings
sys.path.append('../')
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import torch
from src.utils.config                import CFG
from src.preprocessing.label_utils   import load_all_labels
from src.preprocessing.dataset_full  import ECGDatasetFull
from src.models.leadwise_transformer import LeadwiseTransformer, build_leadwise_with_peft
from src.training.train_peft         import run_experiment

DATA_PATH = CFG['data']['path']
device    = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'GPU:   {torch.cuda.get_device_name(0)}')
print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Results dir: {CFG["paths"]["results"]}')

## Data

Same PTB-XL split used in notebook 04: folds 1–8 for training, fold 9 for validation. `ECGDatasetFull` is used — one record per sample, shape `(12, 1000)`. The test fold (10) remains held out.

In [ ]:
Y = load_all_labels(
    DATA_PATH + 'ptbxl_database.csv',
    DATA_PATH + 'scp_statements.csv',
)
train_df = Y[Y.strat_fold <  9]
val_df   = Y[Y.strat_fold == 9]

train_ds = ECGDatasetFull(train_df, DATA_PATH)
val_ds   = ECGDatasetFull(val_df,   DATA_PATH)

print(f'Train: {len(train_df):,} records -> {len(train_ds):,} samples')
print(f'Val:   {len(val_df):,} records  -> {len(val_ds):,} samples')

## Architecture Verification

Before training, we verify:
1. **Shape** — `(B, 12, 1000)` input produces `(B, 5)` logits
2. **Parameter count** — LoRA r=8 keeps trainable fraction below 10% of the base model

Parameter counts are computed dynamically from the model itself — no hardcoded totals.

In [ ]:
_dummy = torch.randn(4, 12, 1000)

# Base model stats
base   = LeadwiseTransformer()
bp     = base.count_parameters()
with torch.no_grad():
    out = base(_dummy)
assert out.shape == (4, 5), f'Expected (4, 5), got {out.shape}'
print(f'Base LeadwiseTransformer')
print(f'  Trainable: {bp["trainable"]:,} / {bp["total"]:,} ({bp["percentage"]})')
print(f'  Forward:   {tuple(_dummy.shape)} -> {tuple(out.shape)} -- OK')
del base, out

# LoRA r=8 verification
lora8   = build_leadwise_with_peft(rank=8, use_dora=False)
p_lora8 = lora8.count_parameters()
pct     = p_lora8['trainable'] / p_lora8['total']
assert pct < 0.10, f"LoRA trainable {pct:.1%} exceeds 10% -- check target_modules"
with torch.no_grad():
    out = lora8(_dummy)
assert out.shape == (4, 5)
print(f'\nLoRA r=8')
print(f'  Trainable: {p_lora8["trainable"]:,} / {p_lora8["total"]:,} ({p_lora8["percentage"]}) -- OK')
print(f'  Forward:   {tuple(_dummy.shape)} -> {tuple(out.shape)} -- OK')
del lora8, out, _dummy

## Training: LeadwiseTransformer + LoRA r=8

LoRA adapters are injected into the attention Q and V projections of both the temporal Transformer encoder and the cross-lead attention layer. The base model weights are frozen; only adapter weights and the classifier head are updated.

Because LeadwiseTransformer is trained from scratch (no pretrained backbone), a higher learning rate (`lr_peft` from config) is used compared to the HuBERT experiments. Training uses the same `run_experiment` loop as notebook 04: warmup LR schedule, BCEWithLogitsLoss with class-imbalance `pos_weight`, and patience-based early stopping.

In [ ]:
model_lora = build_leadwise_with_peft(rank=8, use_dora=False)

auc_lora, hist_lora = run_experiment(
    model_lora, train_ds, val_ds,
    experiment_name='leadwise_lora_r8',
    epochs=CFG['training']['epochs'],
    lr=CFG['training']['lr_peft'],
    batch_size=CFG['training']['batch_size_full'],
    save_dir=CFG['paths']['results'],
)
del model_lora; torch.cuda.empty_cache()

print(f'\nLeadwise LoRA r=8: AUC {auc_lora:.4f}')
print(f'Full comparison and plots -> 04c_comparison.ipynb')